# Weld Defect Analyzer — v2 (Preprocessed) GPU Training

Trains `weld_v2_preprocessed` (YOLOv8n-cls) on the RIAWELC dataset **on a free GPU** (Colab T4 / Kaggle).
Full 50 epochs finish in ~1–2 hours instead of ~71 hours on CPU.

### One-time setup before running
1. On your PC, zip the raw dataset folder `C:\Users\nisha\Downloads\DB - Copy` → `DB - Copy.zip`.
2. Upload `DB - Copy.zip` to the **root of your Google Drive** (My Drive).
3. In Colab: **Runtime → Change runtime type → GPU (T4)**.
4. Run the cells top to bottom.

At the end you download `weld_v2_best.zip`. Unzip it into the repo at:
`runs/classify/weld_v2_preprocessed/weights/best.pt` — the Streamlit app's *Preprocessed Model (weld_v2)* selector then works.

In [ ]:
# 1) Verify GPU + install deps
import subprocess, torch
print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Runtime > Change runtime type > GPU, then re-run.')
subprocess.run(['pip', 'install', '-q', 'ultralytics>=8.3', 'opencv-python-headless', 'tqdm'], check=True)
print('Deps installed.')

In [ ]:
# 2) Mount Drive + config
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ZIP    = '/content/drive/MyDrive/DB - Copy.zip'  # <- adjust if you named/placed it differently
WORK         = '/content/weld'
RAW_ROOT     = f'{WORK}/raw'
PROCESSED    = f'{WORK}/data/processed'
PREPROCESSED = f'{WORK}/data/preprocessed'
RUN_PROJECT  = f'{WORK}/runs/classify'
RUN_NAME     = 'weld_v2_preprocessed'

import os
assert os.path.exists(DRIVE_ZIP), f'Zip not found at {DRIVE_ZIP} — upload DB - Copy.zip to My Drive root.'
print('Found dataset zip:', DRIVE_ZIP)

In [ ]:
# 3) Unzip raw dataset and locate the split folders
import os, zipfile, shutil
shutil.rmtree(RAW_ROOT, ignore_errors=True)
os.makedirs(RAW_ROOT, exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP) as z:
    z.extractall(RAW_ROOT)

# Find the directory that actually contains training/validation/testing
RAW_BASE = None
for root, dirs, _ in os.walk(RAW_ROOT):
    low = {d.lower() for d in dirs}
    if {'training', 'validation', 'testing'}.issubset(low):
        RAW_BASE = root
        break
assert RAW_BASE, 'Could not find training/validation/testing folders inside the zip.'
print('Raw dataset base:', RAW_BASE)

In [ ]:
# 4) Map RIAWELC -> YOLO classification layout (data/processed)
import os, shutil
SPLIT_MAP = {'training': 'train', 'validation': 'val', 'testing': 'test'}
CLASS_MAP = {'Difetto1': 'CR', 'Difetto2': 'LP', 'Difetto4': 'PO', 'NoDifetto': 'ND'}

shutil.rmtree(PROCESSED, ignore_errors=True)
total = 0
for raw_split, yolo_split in SPLIT_MAP.items():
    for raw_class, yolo_class in CLASS_MAP.items():
        src = os.path.join(RAW_BASE, raw_split, raw_class)
        dst = os.path.join(PROCESSED, yolo_split, yolo_class)
        if not os.path.isdir(src):
            print('[SKIP]', src); continue
        os.makedirs(dst, exist_ok=True)
        files = [f for f in os.listdir(src) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        for f in files:
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
        total += len(files)
        print(f'[OK] {raw_split}/{raw_class} -> {yolo_split}/{yolo_class} ({len(files)})')
print(f'\nMapped {total} images into {PROCESSED}')

In [ ]:
# 5) Preprocess: CLAHE (local contrast) + non-local-means denoise -> data/preprocessed
#    Same pipeline as src/preprocess.py, parallelized across CPU cores.
import os, cv2, numpy as np
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

def _preprocess_one(in_path, rel, out_dir):
    try:
        out_path = Path(out_dir) / rel
        out_path.parent.mkdir(parents=True, exist_ok=True)
        img = cv2.imread(in_path)
        if img is None:
            return False
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img = cv2.cvtColor(clahe.apply(gray), cv2.COLOR_GRAY2BGR)
        img = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)
        cv2.imwrite(str(out_path), img)
        return True
    except Exception as e:
        print('[ERR]', in_path, e); return False

src_root = Path(PROCESSED)
files = [p for p in src_root.rglob('*') if p.suffix.lower() in ('.png', '.jpg', '.jpeg')]
print(f'Preprocessing {len(files)} images (CLAHE + denoise)...')
shutil.rmtree(PREPROCESSED, ignore_errors=True)
ok = 0
with ProcessPoolExecutor() as ex:
    futs = [ex.submit(_preprocess_one, str(p), str(p.relative_to(src_root)), PREPROCESSED) for p in files]
    for i, f in enumerate(as_completed(futs), 1):
        ok += bool(f.result())
        if i % 2000 == 0 or i == len(files):
            print(f'  {i}/{len(files)} (ok={ok})')
print(f'Done. {ok}/{len(files)} preprocessed -> {PREPROCESSED}')

In [ ]:
# 6) Train weld_v2_preprocessed on GPU (50 epochs)
from ultralytics import YOLO
model = YOLO('yolov8n-cls.pt')
results = model.train(
    data     = PREPROCESSED,
    epochs   = 50,
    imgsz    = 224,
    batch    = 64,
    device   = 0,
    project  = RUN_PROJECT,
    name     = RUN_NAME,
    patience = 10,
    plots    = True,
    save     = True,
)
print('\nTraining complete. Best:', f'{RUN_PROJECT}/{RUN_NAME}/weights/best.pt')

In [ ]:
# 7) Validate on the held-out test split + package results for download
import shutil, os
run_dir = f'{RUN_PROJECT}/{RUN_NAME}'
best = f'{run_dir}/weights/best.pt'

metrics = YOLO(best).val(data=PREPROCESSED, split='test')
print('Test top-1 acc:', getattr(metrics, 'top1', 'n/a'))

out_zip = '/content/weld_v2_best'
shutil.make_archive(out_zip, 'zip', run_dir)  # zips weights/ + plots + results.csv
print('Packaged:', out_zip + '.zip')
from google.colab import files
files.download(out_zip + '.zip')

## Put the trained model back into the repo

Unzip `weld_v2_best.zip` and copy its contents into the repo so the structure is:

```
runs/classify/weld_v2_preprocessed/
    weights/best.pt          <- required by the app
    results.csv
    confusion_matrix.png ...
```

The Streamlit app already looks for `runs/classify/weld_v2_preprocessed/weights/best.pt`
(see `load_predictor` in `src/app.py`), so once the file is there the
**Preprocessed Model (weld_v2)** option works locally and on Streamlit Cloud.

> Note: `runs/` and `*.pt` are gitignored. To deploy v2 on Streamlit Cloud you must
> force-add the weight: `git add -f runs/classify/weld_v2_preprocessed/weights/best.pt`.